# Chapter 14: Sorting Without Comparing

The programs from Chapter 14 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## The Speed Limit for Comparison Sorts

`lower_bound.py`

In [1]:
"""Why comparison sorts need about n log2 n comparisons: there are n!
possible orders, and each yes-or-no comparison can at best halve the
orders still possible, so it takes at least log2(n!) comparisons."""

import math

print(f"{'n':>7} {'log2(n!)':>11} {'n log2 n':>11}")
for n in [10, 100, 1_000, 10_000, 100_000]:
    log2_factorial = math.lgamma(n + 1) / math.log(2)  # log2 of n!
    print(f"{n:>7,} {log2_factorial:>11,.0f} {n * math.log2(n):>11,.0f}")

      n    log2(n!)    n log2 n
     10          22          33
    100         525         664
  1,000       8,529       9,966
 10,000     118,458     132,877
100,000   1,516,704   1,660,964


## Counting Sort

`linear_sorts.py`

In [2]:
"""Sorts that never compare two items, for small non-negative integers."""


def counting_sort(numbers, largest):
    """Sort integers from 0 to largest by counting each value:
    time and memory proportional to n + largest."""
    counts = [0] * (largest + 1)
    for x in numbers:
        counts[x] += 1
    out = []
    for value, count in enumerate(counts):
        out.extend([value] * count)
    return out


def counting_sort_by(items, key, largest):
    """A stable counting sort of any items by an integer key."""
    counts = [0] * (largest + 1)
    for item in items:
        counts[key(item)] += 1
    starts, total = [0] * (largest + 1), 0
    for value in range(largest + 1):  # where each key's items begin
        starts[value], total = total, total + counts[value]
    out = [None] * len(items)
    for item in items:  # in input order: equal keys stay in order
        k = key(item)
        out[starts[k]] = item
        starts[k] += 1
    return out


def radix_sort(numbers, base=256):
    """Sort non-negative integers digit by digit, least significant
    digit first, with a stable counting sort for each digit."""
    if not numbers:
        return []
    out, place = list(numbers), 1
    while place <= max(numbers):
        out = counting_sort_by(out, lambda x: x // place % base, base - 1)
        place *= base
    return out

## Radix Sort

`radix_demo.py`

In [3]:
"""Radix sort in base 10, one digit at a time."""

from linear_sorts import counting_sort_by

numbers = [170, 45, 75, 90, 802, 24, 2, 66]
place = 1
while place <= max(numbers):
    numbers = counting_sort_by(numbers, lambda x: x // place % 10, 9)
    print(f"by the {place}s digit: {numbers}")
    place *= 10

by the 1s digit: [170, 90, 802, 2, 24, 45, 75, 66]
by the 10s digit: [802, 2, 24, 45, 66, 170, 75, 90]
by the 100s digit: [2, 24, 45, 66, 75, 90, 170, 802]


## The Benchmark's Lesson

`bench_linear.py`

In [4]:
"""Our counting and radix sorts against sorted(), on n numbers from 0
to 999. Fewer steps in theory; but sorted() runs in C."""

import random

import bench
from linear_sorts import counting_sort, radix_sort

rng = random.Random(2026)


def data(n):
    return ([rng.randint(0, 999) for _ in range(n)],)


sizes = [100_000, 200_000, 400_000, 800_000]
results = {
    "counting_sort": bench.doubling(
        lambda d: counting_sort(d, 999), data, sizes
    ),
    "radix_sort": bench.doubling(radix_sort, data, sizes),
    "sorted()": bench.doubling(sorted, data, sizes),
}
bench.table(results)

         n   counting_sort      radix_sort        sorted()
   100,000         2.27 ms         29.8 ms         12.5 ms
   200,000    4.48 ms x2.0    61.5 ms x2.1    25.7 ms x2.1
   400,000    8.61 ms x1.9     129 ms x2.1    54.5 ms x2.1
   800,000      18 ms x2.1     278 ms x2.2     110 ms x2.0
